In [ ]:
import os
from pathlib import Path

from langchain.chat_models import init_chat_model
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Ingestion Process

In [ ]:
# Load Document — resolve `data` whether cwd is repo root or rag_practice/
_cwd = Path.cwd().resolve()
print(_cwd)
if (_cwd / "data").is_dir():
    ROOT_PATH = _cwd / "data"
elif (_cwd.parent / "data").is_dir():
    ROOT_PATH = _cwd.parent / "data"
else:
    ROOT_PATH = _cwd / "data"
doc_list = ["kl_tourism.pdf","Chennai_data.docx", "hyd_data.pdf", "pu_tourism.txt"]


def parse_document(docs: list[str]) -> list[Document]:
    all_docs:list[Document] = []
    for doc in docs:
        file_path = ROOT_PATH / doc
        if not file_path.is_file():
            print(f"File Path Issue: {file_path}, continuing with the rest of file")
            continue
        _, ext = os.path.splitext(doc.lower())
        if ext == ".txt":
            all_docs.extend(extract_content(TextLoader(file_path)))
        elif ext == ".pdf":
            all_docs.extend(extract_content(PyPDFLoader(file_path)))
    return all_docs


def extract_content(loader: BaseLoader) -> list[Document]:
    docs:list[Document]= []
    try:
        docs = loader.load()
        print(f"Content Type: {type(loader).__name__} / Content Length: {len(docs)}")
    except Exception as e:
        print(f"Extraction Error! {e}")
    return docs

merged_docs = parse_document(doc_list)
if not merged_docs:
    print("No Docs available to ingest")
else:
    print(f"Total Docs size: {len(merged_docs)}")

In [ ]:
#Text Splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunked_docs:list[Document] = splitter.split_documents(merged_docs)
print(len(chunked_docs))

In [ ]:
#Define Embeddings & Collection name
doc_embedding_model = OllamaEmbeddings(model="embeddinggemma:300m",base_url="http://127.0.0.1:11434",)
db_collection_name = "travel_docs"

### Chroma DB

In [ ]:
# Create vector DB and store docs
from langchain_chroma.vectorstores import Chroma

store_directory = (ROOT_PATH / "db").resolve()
os.makedirs(store_directory, exist_ok=True)

persist_path = store_directory.as_posix()

#Initialize Chroma DB
chroma_vector_db = Chroma(collection_name=db_collection_name, embedding_function=doc_embedding_model, persist_directory=persist_path)


In [ ]:
## Quick for prototype
# chroma_vector_db = Chroma.from_documents(
#     documents=chunked_docs,
#     embedding=doc_embedding_model,
#     collection_name="travel_docs",
#     persist_directory=persist_path
#)

In [ ]:
#add docs to the chroma db
ids = chroma_vector_db.add_documents(documents=chunked_docs)
print(len(ids))
chroma_vector_db.get()["ids"][1]
#chroma_vector_db.get("4fa5d9e0-9758-4730-b712-5061c312ffe9")

In [ ]:
result = chroma_vector_db.get()
#print(result)
for info in result:
    print(info)

print(f"{len(result["documents"])}")


In [ ]:
response = chroma_vector_db.similarity_search_with_score("tell me something about pondicherry beach?", k=2)
# #print(response)
# for doc_res, score in response:
#     print(doc_res, score)

### Retrieval

In [ ]:
# Low Level retrieval
prompt = ChatPromptTemplate.from_template("""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:""")

llm = init_chat_model(model="llama3.2:latest", model_provider="ollama", temperature="0.0")

while True:
    usr_input = str(input("User: "))
    if usr_input == "/bye":
        print("Thanks for using this travel info. bot")
        break
    response = chroma_vector_db.similarity_search_with_score(usr_input, k=3)
    context = "\n\n".join(doc.page_content for doc, score in response)
    chain = prompt | llm | StrOutputParser()
    agent_result = chain.invoke({"context": context, "question": usr_input})
    print(f"AI: {agent_result}")


In [ ]:
# Abstract Layer with as_Retrieval
retriever = chroma_vector_db.as_retriever(search_type="similarity",
                                           search_kwargs={"k":3})#, "score_threshold":0.2})
#retriever = chroma_vector_db.as_retriever()
#retriever_response = retriever.invoke("tell me something about pondicherry beach?")
#print(retriever_response)

retriever_prompt = ChatPromptTemplate.from_template("""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:""")

llm = init_chat_model(model="llama3.2:latest", model_provider="ollama", temperature="0.0")

def form_docs(docs:list[Document])->str:
    return "\n\n".join(doc.page_content for doc in docs)

while True:
    usr_input = str(input("User: "))
    if usr_input == "/bye":
        print("Thanks for using this travel info. bot")
        break
    retriever_chain = ({"context":retriever | form_docs, "question":RunnablePassthrough()} | retriever_prompt | llm | StrOutputParser())
    agent_result = retriever_chain.invoke(usr_input)
    print(f"AI: {agent_result}")



In [78]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

#Prebuild Retriever
pre_built_retriever = retriever = chroma_vector_db.as_retriever(search_type="similarity",
                                           search_kwargs={"k":3})#, "score_threshold":0.2})
pre_built_retriever_prompt = ChatPromptTemplate.from_template("""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {input}
Context: {context}
Answer:""")
llm = init_chat_model(model="llama3.2:latest", model_provider="ollama", temperature="0.0")

question_answer_chain = create_stuff_documents_chain(llm, pre_built_retriever_prompt)
chain_qa = create_retrieval_chain(pre_built_retriever, question_answer_chain)
response = chain_qa.invoke({"input": "where is auroville?"})
print(response["answer"])

#Direct Usage of RetrievalQA - Depreacted
# chain_qa = RetrievalQA.from_chain_type(retriever = pre_built_retriever, chain_type="stuff", llm=llm)
# chain_qa.invoke("who is the president of india before 2023?")

Auroville is located near Pondicherry across parts of Tamil Nadu territory. It is not entirely inside Pondicherry, but rather situated nearby. Auroville is an international experimental township founded in 1968 with the vision of human unity and sustainable living.


### Qdrant DB

In [ ]:
from langchain_qdrant import QdrantVectorStore
from langchain_qdrant.vectorstores import QdrantClient
from qdrant_client.http.models import VectorParams, Distance

# Qdrant Client & Create Collection
qdrant_client = QdrantClient(host="localhost")
if not qdrant_client.collection_exists(collection_name=db_collection_name):
    qdrant_client.create_collection(collection_name=db_collection_name, vectors_config=VectorParams(size=768, distance=Distance.COSINE))

# Create Qdrant Vector Store with the collection
qd_vector_store = QdrantVectorStore(client=qdrant_client, collection_name=db_collection_name, embedding=doc_embedding_model)

## Quick Implementations without explicit client
# QdrantVectorStore.from_documents(
#         documents=chunked_docs,
#         embedding=doc_embedding_model,
#         collection_name=db_collection_name,
#         url="http://localhost:6333"
#     )

In [ ]:
# Add Data -- Verify the Duplicate logics -- TODO:
qd_vector_store.add_documents(documents=chunked_docs)

In [ ]:
result = qd_vector_store.similarity_search("tell about aurovile?", k=3)
print(len(result))
result